In [1]:
import glob
import pandas as pd

# Load files

In [2]:
files = glob.glob("data/results_*.jsonl")
files

['data/results_refugee.jsonl',
 'data/results_unhcr.jsonl',
 'data/results_prwp.jsonl']

In [3]:
data = pd.concat([pd.read_json(x, lines=True) for x in files]).reset_index(drop=True)
data

,snapshot_id,source,model,raw_output,usage,cost,status,incomplete_details,parsed_output,error
0,027_Jordan-Emergency-Food-Security-Project_fig...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 7168, 'input_tokens_details':...","{'input_tokens': 7168, 'output_tokens': 2431, ...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
1,054_Sudan-Basic-Education-Emergency-Support-Pr...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 13799, 'input_tokens_details'...","{'input_tokens': 13799, 'output_tokens': 1576,...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
2,134_PAD7340ARABIC00n0220020140Arabic01_figure_...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 13286, 'input_tokens_details'...","{'input_tokens': 13286, 'output_tokens': 1460,...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
3,004_BOSIB-87c444de-4797-4bf9-b654-4932a7fb0112...,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""figure_title"",""o...","{'input_tokens': 8299, 'input_tokens_details':...","{'input_tokens': 8299, 'output_tokens': 2116, ...",completed,NaN,"{'fields': [{'metadata_field': 'figure_title',...",NaN
4,202_multi0page_figure_000.png,refugee,gpt-5.5,"{""fields"":[{""metadata_field"":""geographic_scope...","{'input_tokens': 12161, 'input_tokens_details'...","{'input_tokens': 12161, 'output_tokens': 2134,...",completed,NaN,{'fields': [{'metadata_field': 'geographic_sco...,NaN
...,...,...,...,...,...,...,...,...,...,...
205,document_8886604_table_005.png,prwp,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 12587, 'input_tokens_details'...","{'input_tokens': 12587, 'output_tokens': 1475,...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN
206,document_6284707_table_004.png,prwp,gpt-5.5,"{""fields"":[{""metadata_field"":""table_panel_titl...","{'input_tokens': 8238, 'input_tokens_details':...","{'input_tokens': 8238, 'output_tokens': 3145, ...",completed,NaN,{'fields': [{'metadata_field': 'table_panel_ti...,NaN
207,document_7095002_table_002.png,prwp,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 9890, 'input_tokens_details':...","{'input_tokens': 9890, 'output_tokens': 1640, ...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN
208,document_7063025_table_001.png,prwp,gpt-5.5,"{""fields"":[{""metadata_field"":""table_title"",""ob...","{'input_tokens': 8278, 'input_tokens_details':...","{'input_tokens': 8278, 'output_tokens': 1525, ...",completed,NaN,"{'fields': [{'metadata_field': 'table_title', ...",NaN


In [4]:
data[data["error"].notna()]

,snapshot_id,source,model,raw_output,usage,cost,status,incomplete_details,parsed_output,error


In [5]:
data["snapshot_type"] = data["snapshot_id"].apply(lambda x: x.split("_")[-2])
data.groupby(["source", "snapshot_type"]).agg(
    count=("source", "count"),
)

count
source  snapshot_type       
prwp    figure            35
        table             35
refugee figure            35
        table             35
unhcr   figure            35
        table             35

# Preprocessing

In [6]:
pd.set_option('display.max_colwidth', None)

In [7]:
df = data[["snapshot_id", "snapshot_type", "source", "model", "parsed_output"]].copy()
df["parsed_output"] = df["parsed_output"].apply(
    lambda x: x.get("fields", []) if isinstance(x, dict) else []
)

df = df.explode("parsed_output").reset_index(drop=True)
df = df.join(pd.json_normalize(df["parsed_output"])).drop(columns="parsed_output")

df

,snapshot_id,snapshot_type,source,model,metadata_field,observed_value,description,source_level,discovery_value,reasoning
0,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,figure_title,Figure 2: Share of food expenditure in total expenditure p.c. (%),The title or caption identifying the figure and its subject.,snapshot,high,The figure title provides the most direct searchable description of the snapshot content.
1,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,indicator_name,Share of food expenditure in total expenditure p.c.,The main measure or indicator represented in the figure.,snapshot,high,Indicator-level metadata enables retrieval of figures by the specific economic or food-security measure shown.
2,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,subject_domain,Food expenditure; food security,The thematic domain or topical area represented by the snapshot.,both,high,The figure title and document metadata both connect the snapshot to food expenditure and food security topics.
3,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,geographic_scope,"Jordan, inferred from parent document title and project name; not visible in snapshot","The country, region, or geographic area to which the figure data applies.",document,high,Geographic context is essential for discovery because it is not shown in the figure itself.
4,027_Jordan-Emergency-Food-Security-Project_figure_001.png,figure,refugee,gpt-5.5,time_period,Not identifiable from this snapshot,"The date, year, or period covered by the data in the figure.",snapshot,high,"A data time period would be highly useful for comparison, but it is not visible in the figure."
...,...,...,...,...,...,...,...,...,...,...
3036,document_10395051_table_002.png,table,prwp,gpt-5.5,significance_notation,"*, ** and *** denote significance at the 10 percent, 5 percent and 1 percent levels, respectively",Notation used to indicate statistical significance levels.,snapshot,medium,Significance notation is needed to interpret coefficient annotations consistently.
3037,document_10395051_table_002.png,table,prwp,gpt-5.5,robustness_adjustment,Heteroskedasticity-robust t(z)-statistics are presented below the corresponding coefficients,Adjustments or corrections applied to reported statistical inference.,snapshot,medium,Inference adjustments improve reuse by indicating how uncertainty estimates were calculated.
3038,document_10395051_table_002.png,table,prwp,gpt-5.5,endogenous_variable,Each of four informality measures,Variables treated as endogenous in instrumental-variable models.,snapshot,medium,Endogeneity treatment is important for understanding causal empirical specifications.
3039,document_10395051_table_002.png,table,prwp,gpt-5.5,instrumental_variables,Law and order; Business regulatory freedom; Average Years of Secondary Schooling; Sociodemographic factors,Instruments used in instrumental-variable regressions.,snapshot,high,Instrumental variables are key metadata for discovering and evaluating IV regression tables.


In [8]:
df.to_csv("discovery_results.csv", index=False)